# Spintronic Neuromorphic Computing — Google Colab Models
Reduced computational and equivalent-circuit models for MTJ neurons/synapses, STDP, SOT devices, domain walls, oscillator reservoirs, skyrmion reservoirs, p-bits/Ising networks, and hybrid spintronic-CMOS circuits.

These are research compact models, not exact reproductions. Insert paper-specific material/device parameters for quantitative reproduction.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import expit
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.datasets import make_blobs
from sklearn.metrics import accuracy_score
rng=np.random.default_rng(7)
plt.rcParams["figure.figsize"]=(8,4.8)
plt.rcParams["axes.grid"]=True

## 1. MTJ stochastic neuron
\[
P_{sw}=1-\exp[-\Gamma t],\quad \Gamma=f_0e^{-\Delta_{eff}},
\qquad I_{read}=V_{read}G(s).
\]

In [ ]:
RP=10e3; TMR=1.5; RAP=RP*(1+TMR); GP,GAP=1/RP,1/RAP
f0=1e9; Delta0=18
def mtj_G(s): return np.where(np.asarray(s)>0,GP,GAP)
def psw(I,t,Ic=40e-6):
    x=np.clip(np.abs(I)/Ic,0,.999999)
    return 1-np.exp(-f0*np.exp(-Delta0*(1-x)**2)*t)
I=np.linspace(0,80e-6,400)
plt.plot(I*1e6,psw(I,2e-9)); plt.xlabel("I (µA)"); plt.ylabel("Psw"); plt.title("MTJ switching probability"); plt.show()
dt=1e-11; t=np.arange(0,20e-9,dt); Iin=20e-6+25e-6*(np.sin(2*np.pi*.12e9*t)>0)
s=-np.ones_like(t)
for n in range(1,len(t)): s[n]=-s[n-1] if rng.random()<psw(Iin[n],dt) else s[n-1]
plt.plot(t*1e9,s); plt.xlabel("time (ns)"); plt.ylabel("state"); plt.title("Stochastic MTJ neuron"); plt.show()

## 2. MTJ stochastic synapse and STDP
\[
\Delta w=A_+e^{-\Delta t/\tau_+}\;(\Delta t>0),\quad
\Delta w=-A_-e^{\Delta t/\tau_-}\;(\Delta t<0).
\]

In [ ]:
def stdp(dt,Aplus=.08,Aminus=.10,taup=.02,taum=.02):
    dt=np.asarray(dt); return np.where(dt>=0,Aplus*np.exp(-dt/taup),-Aminus*np.exp(dt/taum))
lag=np.linspace(-.08,.08,500); plt.plot(lag*1e3,stdp(lag)); plt.axhline(0,lw=1)
plt.xlabel("post-pre delay (ms)"); plt.ylabel("Δw"); plt.title("STDP kernel"); plt.show()

## 3. SOT programmable synapse
\[
\dot w=\frac{F(I_{SOT})-w}{\tau_w},\qquad G(w)=G_{AP}+\frac{w+1}{2}(G_P-G_{AP}).
\]

In [ ]:
t=np.linspace(0,.5,5000); dt=t[1]-t[0]; Ip=.8+.7*np.sin(2*np.pi*3*t)
w=np.zeros_like(t); w[0]=.5
for i in range(1,len(t)): w[i]=np.clip(w[i-1]+dt*(np.tanh(.25*(Ip[i]-.8))-(w[i-1]-.5))/.08,-1,1)
G=GAP+(w+1)/2*(GP-GAP)
plt.plot(t,w); plt.xlabel("time"); plt.ylabel("w"); plt.title("SOT synapse"); plt.show()
plt.plot(t,G*1e6); plt.xlabel("time"); plt.ylabel("G (µS)"); plt.title("Electrical conductance"); plt.show()

## 4. Hybrid spintronic-CMOS LIF circuit
\[
C_m\dot V=-\frac{V-V_L}{R_m}+I_{syn},\qquad V\ge V_{th}\to spike/reset.
\]

In [ ]:
C=50e-12; Rm=100e6; Vth=.2
t=np.linspace(0,.2,20000); dt=t[1]-t[0]; I=np.zeros_like(t)
for tt in [.03,.031,.032,.08,.081,.12,.121,.122,.16]: I+=3e-12*np.exp(-(t-tt)**2/(2*(2e-4)**2))
V=np.zeros_like(t); sp=[]
for i in range(1,len(t)):
    V[i]=V[i-1]+dt*((-V[i-1]/Rm+I[i])/C)
    if V[i]>=Vth: sp.append(t[i]); V[i]=0
plt.plot(t,V); plt.scatter(sp,[Vth]*len(sp),marker='|',s=140); plt.xlabel('time'); plt.ylabel('V'); plt.title('LIF circuit'); plt.show()

## 5. Domain-wall/all-spin synapse
\[
\dot q=\mu_q I-(q-q_0)/\tau_q,\quad
w=q/L.
\]

In [ ]:
t=np.linspace(0,1,4000); dt=t[1]-t[0]; Ip=.9+.7*(np.sin(2*np.pi*2*t)>0)
q=np.zeros_like(t); q[0]=.3
for i in range(1,len(t)): q[i]=np.clip(q[i-1]+dt*(.8*(Ip[i]-.9)-(q[i]-.3)/.12),0,1)
plt.plot(t,q); plt.xlabel('time'); plt.ylabel('q/L'); plt.title('Domain-wall synaptic state'); plt.show()

## 6. Spin-torque oscillator reservoir
\[
\dot\phi_i=\omega_i+\sum_jK_{ij}\sin(\phi_j-\phi_i)+u(t)+\eta_i.
\]
Only the readout is trained:
\[
W_{out}=YX^T(XX^T+\lambda I)^{-1}.
\]

In [ ]:
N=32; t=np.arange(0,6,.002); u=np.sin(2*np.pi*.7*t)+.4*np.sin(2*np.pi*1.3*t)
A=(rng.random((N,N))<.12).astype(float); np.fill_diagonal(A,0); A/=np.maximum(A.sum(1,keepdims=True),1)
omega=2*np.pi*(.7+.5*rng.random(N)); phi=rng.random(N)*2*np.pi; X=np.zeros((N,len(t)))
for k in range(len(t)):
    X[:,k]=np.sin(phi); c=np.sum(A*np.sin(phi[None,:]-phi[:,None]),axis=1)
    phi += .002*(omega+1.8*c+u[k]+.08*rng.normal(size=N))
y=np.roll(np.tanh(u+.5*u**3),-20); warm=300
m=Ridge(alpha=1e-2,fit_intercept=False).fit(X[:,warm:].T,y[warm:])
plt.plot(t[warm:],y[warm:],label='target'); plt.plot(t[warm:],m.predict(X[:,warm:].T),label='reservoir'); plt.legend()
plt.xlabel('time'); plt.ylabel('output'); plt.title('Spin-oscillator reservoir'); plt.show()

## 7. Skyrmion reservoir
Collective-coordinate proxy:
\[
\mathbf G\times(\mathbf v-\mathbf u)+\alpha\mathbf D(\mathbf v-\beta\mathbf u)+\mathbf F_{pin}+\mathbf F_{drive}=0.
\]
Random pinned textures provide nonlinear temporal responses and resistance/magnetization readouts.

In [ ]:
Nsk=40; t=np.arange(0,8,.005); pos=rng.uniform(-1,1,(Nsk,2)); vel=np.zeros_like(pos); X=np.zeros((2*Nsk+3,len(t)))
for k,tt in enumerate(t):
    u=np.array([np.sin(2*np.pi*.35*tt),.6*np.cos(2*np.pi*.19*tt)]); force=.8*u[None,:]-.25*pos
    for i in range(Nsk):
        d=pos[i]-pos; r2=np.sum(d*d,axis=1)+1e-3
        force[i]+=.015*np.sum(d/(r2[:,None]**1.5),axis=0)
    vel=.88*vel+.12*force; pos+=.005*vel
    X[:Nsk,k]=pos[:,0]; X[Nsk:2*Nsk,k]=pos[:,1]; X[2*Nsk,k]=np.sum(np.exp(-np.sum(pos**2,1)/.18))
    X[2*Nsk+1,k]=pos[:,0].mean(); X[2*Nsk+2,k]=pos[:,1].mean()
y=np.roll(np.sin(.7*t)+.4*np.sin(1.9*t),-5); warm=200
m=Ridge(alpha=1e-2,fit_intercept=False).fit(X[:,warm:].T,y[warm:])
plt.plot(t[warm:],y[warm:],label='target'); plt.plot(t[warm:],m.predict(X[:,warm:].T),label='skyrmion reservoir'); plt.legend()
plt.xlabel('time'); plt.ylabel('output'); plt.title('Skyrmion-fabric reservoir'); plt.show()
plt.scatter(pos[:,0],pos[:,1],s=20); plt.axis('equal'); plt.title('Skyrmion texture proxy'); plt.show()

## 8. p-bit / stochastic Ising network
\[
P(s_i=+1)=\frac{1+\tanh(I_i/I_0)}2,\quad I_i=b_i+\sum_jJ_{ij}s_j.
\]
\[
E=-\frac12s^TJs-b^Ts.
\]

In [ ]:
n=12; J=rng.normal(size=(n,n)); J=(J+J.T)/2; np.fill_diagonal(J,0); J/=np.max(np.abs(J)); b=.15*rng.normal(size=n)
s=rng.choice([-1,1],n); E=[]
for k in range(20000):
    i=rng.integers(n); p=.5*(1+np.tanh((b[i]+J[i]@s)/.35)); s[i]=1 if rng.random()<p else -1
    if k%10==0: E.append(-.5*s@J@s-b@s)
plt.plot(np.arange(len(E))*10,E); plt.xlabel('update'); plt.ylabel('E'); plt.title('p-bit Ising network'); plt.show()

## 9. Fukami SOT artificial neuron
Kurenkov et al. demonstrated LIF and STDP-like behavior from SOT switching dynamics in an AFM/FM heterostructure. Compact model:
\[
\dot m=-m/\tau+\kappa I(t),\quad m\ge m_{th}\Rightarrow spike,\;m\to0.
\]

In [ ]:
t=np.linspace(0,1,10000); dt=t[1]-t[0]; pulses=[.1,.12,.14,.35,.36,.7,.72,.74]
I=sum(np.exp(-(t-p)**2/(2*.0015**2)) for p in pulses); m=np.zeros_like(t); sp=[]
for i in range(1,len(t)):
    m[i]=m[i-1]+dt*(-m[i-1]/.035+.025*I[i])
    if m[i]>=.035: sp.append(t[i]); m[i]=0
plt.plot(t,m); plt.scatter(sp,[.035]*len(sp),marker='|',s=140); plt.xlabel('time'); plt.ylabel('m'); plt.title('SOT artificial neuron'); plt.show()

## 10. MTJ differential crossbar
\[
I_j=\sum_iG_{ij}V_i.
\]
Signed weights are represented by differential conductances \(G^+-G^-\).

In [ ]:
X,y=make_blobs(n_samples=1200,centers=3,n_features=6,cluster_std=2,random_state=4)
clf=LogisticRegression(max_iter=1000).fit(X[:900],y[:900]); W=clf.coef_; b=clf.intercept_
score=X[900:]@(np.maximum(W,0)-np.maximum(-W,0)).T+b; pred=np.argmax(score,1)
print('crossbar-equivalent accuracy:',accuracy_score(y[900:],pred))
plt.imshow(W,aspect='auto'); plt.xlabel('input'); plt.ylabel('output'); plt.title('Differential conductance matrix'); plt.show()

## Open-source upgrade path

**mumax3** or **OOMMF/Ubermag**: LLG + STT/SOT, thermal fluctuations, DMI, domain walls, skyrmions.

**ngspice + PySpice**: circuit-level MTJ/LIF/crossbar co-simulation.

Full magnetic dynamics:
\[
\dot m=-\gamma m\times H_{eff}+\alpha m\times\dot m+\tau_{STT}+\tau_{SOT}.
\]

Use exact material/device parameters from the target paper before claiming quantitative reproduction.